# RAG with Ollama, LangChain, and FAISS 🧠 (Optimized and Updated Version)
# This script demonstrates an optimized Retrieval-Augmented Generation (RAG) pipeline
# with improved performance, logging, and progress tracking.

In [1]:
# ==============================================================================
# CELL 1: Setup and Installations
# ==============================================================================
import os
import logging
import re
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Enhanced Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("--- Initializing AI Medical Assistant with Local Ollama Models ---")

2025-08-26 16:50:16,151 - INFO - --- Initializing AI Medical Assistant with Local Ollama Models ---


In [2]:
# ==============================================================================
# CELL 2: Prepare the Knowledge Base from a folder of PDFs
# ==============================================================================
knowledge_base_dir = "knowledge_base"
if not os.path.exists(knowledge_base_dir):
    os.makedirs(knowledge_base_dir)
    logging.warning(f"Created a folder '{knowledge_base_dir}'. Please add your PDF files to it and rerun.")
    exit()

all_chunks = []
pdf_files = [f for f in os.listdir(knowledge_base_dir) if f.endswith('.pdf')]

if not pdf_files:
    logging.error(f"No PDF files found in '{knowledge_base_dir}'. Please add PDFs.")
    exit()

logging.info(f"[Step 1] Loading and splitting documents from '{knowledge_base_dir}'...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

# Use tqdm to show a progress bar while loading and splitting PDFs
for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
    pdf_path = os.path.join(knowledge_base_dir, pdf_file)
    try:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load_and_split(text_splitter=text_splitter)
        all_chunks.extend(docs)
        logging.info(f"Loaded and split {len(docs)} chunks from {pdf_file}")
    except Exception as e:
        logging.error(f"Failed to process {pdf_file}. Error: {e}")

if not all_chunks:
    logging.error("Could not extract any text from the PDF files. Exiting.")
    exit()

logging.info(f"Successfully split documents into a total of {len(all_chunks)} chunks.")

2025-08-26 16:50:16,199 - INFO - [Step 1] Loading and splitting documents from 'knowledge_base'...
Processing PDFs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:30<00:00,  2.16s/it]
2025-08-26 16:50:46,437 - INFO - Successfully split documents into a total of 451 chunks.


In [3]:
# ==============================================================================
# CELL 3: Create or Load Vector Store with Batch Processing
# ==============================================================================
# This cell creates the FAISS vector store from the text chunks generated in
# Cell 2. It is optimized for low-memory systems by processing the documents
# in small batches. It also includes improved logging and a more reliable
# check for an existing vector store.
# ==============================================================================

vector_store_path = "faiss_index_ollama"
faiss_index_file = os.path.join(vector_store_path, "index.faiss")

# --- 1. Initialize Ollama Embeddings ---
embeddings = OllamaEmbeddings(model="all-minilm")

# --- 2. Check for Existing Vector Store ---
# We check for the actual 'index.faiss' file, which is more reliable than
# just checking if the directory exists.
if os.path.exists(faiss_index_file):
    logging.info(f"[Step 2] Loading existing vector store from '{vector_store_path}'...")
    try:
        vector_store = FAISS.load_local(
            vector_store_path,
            embeddings,
            allow_dangerous_deserialization=True # Required for loading FAISS indexes
        )
        logging.info("Vector store loaded successfully!")
    except Exception as e:
        logging.error(f"Failed to load vector store. Error: {e}")
        logging.error("Consider deleting the existing index folder and rerunning.")
        exit()
else:
    # --- 3. Create New Vector Store in Batches ---
    logging.info("[Step 2] No existing index found. Creating new vector store...")
    logging.info("This may take a while depending on the number of documents.")

    try:
        # Define the size of each batch to process.
        BATCH_SIZE = 32
        
        # The first batch creates the vector store.
        logging.info(f"--> Processing initial batch of {min(BATCH_SIZE, len(all_chunks))} chunks...")
        first_batch = all_chunks[:BATCH_SIZE]
        vector_store = FAISS.from_documents(documents=first_batch, embedding=embeddings)
        
        # Subsequent batches are added to the existing store.
        num_batches = (len(all_chunks) - 1) // BATCH_SIZE + 1
        for i in range(1, num_batches):
            start_index = i * BATCH_SIZE
            end_index = start_index + BATCH_SIZE
            next_batch = all_chunks[start_index:end_index]
            
            logging.info(f"--> Processing batch {i+1}/{num_batches} ({len(next_batch)} chunks)...")
            vector_store.add_documents(documents=next_batch)

        # Save the completed vector store to disk.
        logging.info("All batches processed. Saving vector store to disk...")
        vector_store.save_local(vector_store_path)
        logging.info(f"Vector store created and saved to '{vector_store_path}'!")

    except Exception as e:
        logging.error(f"Failed to create vector store. Error: {e}")
        logging.error(f"Please ensure Ollama is running and you have pulled the '{embeddings.model}' model.")
        exit()

2025-08-26 16:50:46,663 - INFO - [Step 2] Loading existing vector store from 'faiss_index_ollama'...
2025-08-26 16:50:46,673 - INFO - Loading faiss.
2025-08-26 16:50:46,743 - INFO - Successfully loaded faiss.
2025-08-26 16:50:46,832 - INFO - Vector store loaded successfully!


In [4]:
# ==============================================================================
# CELL 4: Build the Improved RAG and Non-RAG Chains
# ==============================================================================

# --- 1. Define Retriever and Model ---
retriever = vector_store.as_retriever()
# 'tinyllama' is a great choice for performance on limited hardware.
model = ChatOllama(model="tinyllama")

# --- 2. Create the New, Simplified Prompt Template ---
# This prompt is much more direct. It clearly defines the role, the context,
# the task, and the desired output format.
template = """
You are an AI medical assistant. Your task is to help a patient organize their thoughts before a doctor's visit.
Based on the provided medical context and the patient's symptom, generate exactly 5 questions to ask the patient about their medical history. These questions should help the patient reflect and structure their information effectively for their doctor.

MEDICAL CONTEXT:
{context}

PATIENT'S SYMPTOM:
{symptom}

YOUR 5 QUESTIONS FOR THE PATIENT:
"""

prompt = ChatPromptTemplate.from_template(template)

# --- 3. Build the RAG Chain ---
rag_chain = (
    {
        "context": retriever,
        "symptom": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

# --- 4. Build the Non-RAG Chain for comparison ---
# This chain uses the same prompt but passes an empty string for context.
non_rag_chain = (
    {
        "context": lambda x: "", # Always provide empty context
        "symptom": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

logging.info("[Step 3] AI Assistant RAG and Non-RAG chains are ready.")

2025-08-26 16:50:47,279 - INFO - [Step 3] AI Assistant RAG and Non-RAG chains are ready.


In [ ]:
# ==============================================================================
# CELL 5: Ask Questions and Get Actionable Results
# ==============================================================================

def get_questions_for_symptom(symptom):
    """Invokes both RAG and non-RAG chains for a given symptom and prints the results."""
    if not vector_store:
        logging.error("Vector store is not available. Cannot process symptom.")
        return
        
    logging.info(f"Processing symptom: '{symptom}'")
    print(f"\n{'='*80}")
    print(f"PATIENT'S SYMPTOM: {symptom}")
    print(f"{'-'*80}")

    # --- Non-RAG Invocation ---
    print("\n🤖 Generating questions WITHOUT RAG (from model's general knowledge)...")
    try:
        non_rag_response = non_rag_chain.invoke(symptom)
        questions = re.findall(r'\d+\.\s(.*?)(?=\n\d+\.|\Z)', non_rag_response, re.DOTALL)
        if questions:
            for i, q in enumerate(questions, 1):
                print(f"{i}. {q.strip()}")
        else:
            print("Could not parse questions. Raw response:\n", non_rag_response)
    except Exception as e:
        logging.error(f"An error occurred during Non-RAG chain invocation: {e}")

    print(f"\n{'-'*40}\n")

    # --- RAG Invocation ---
    print("📚 Generating questions WITH RAG (guided by your documents)...")
    try:
        rag_response = rag_chain.invoke(symptom)
        questions = re.findall(r'\d+\.\s(.*?)(?=\n\d+\.|\Z)', rag_response, re.DOTALL)
        if questions:
            for i, q in enumerate(questions, 1):
                print(f"{i}. {q.strip()}")
        else:
            print("Could not parse questions. Raw response:\n", rag_response)
    except Exception as e:
        logging.error(f"An error occurred during RAG chain invocation: {e}")

    print(f"\n{'='*80}")

# --- Start the AI Assistant Session ---
print("\n--- AI Medical Assistant Session ---")

# Test with the same symptoms from your original notebook
get_questions_for_symptom("My right knee hurts.")
get_questions_for_symptom("I am feeling tired all the time.")
get_questions_for_symptom("I was crouched and when I rised up, I got dizzy and my vision went black for about 15 seconds.")

logging.info("--- Session finished ---")

2025-08-26 16:50:47,345 - INFO - Processing symptom: 'My right knee hurts.'



--- AI Medical Assistant Session ---

PATIENT'S SYMPTOM: My right knee hurts.
--------------------------------------------------------------------------------

🤖 Generating questions WITHOUT RAG (from model's general knowledge)...


2025-08-26 17:03:46,918 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


1. When did your symptom start? Answer: "I don't have a specific start date, but it started gradually over the past few weeks."
2. What was the first sign of the problem? Answer: "The pain has been increasing in intensity over the last few days."
3. How does the pain feel when you feel it now? Answer: "It's sharp and intense right now, but it may become dull or even numb over time."
4. What are your current symptoms like? Answer: "Your knee is uncomfortable, but you haven't had any other problems like numbness or pain in the leg."
5. Have you ever had a similar problem before? Answer: "I have heard of some knee injuries that cause this kind of pain. Is there anything else you can think of?"
6. Have you experienced any medical treatments or procedures recently? Answer: "No, I haven't seen a doctor or received any special treatment for my knee yet."
7. Can you describe the location and size of your knee? Answer: "It's about 5 inches off the ground at the front of my leg, but it's not too

2025-08-26 17:17:44,386 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-08-26 17:35:47,722 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-08-26 17:37:14,885 - INFO - Processing symptom: 'I am feeling tired all the time.'


...

1. When did you last feel energetic or alert?
2. Have you ever had a fever before?
3. Do you have any underlying medical conditions that could affect your sleep or appetite?
4. Are there any medications or supplements you take, and how often do they work for you?
5. Do you have any allergies or sensitivities that might cause you to feel unwell?

----------------------------------------

📚 Generating questions WITH RAG (guided by your documents)...


2025-08-26 17:59:36,087 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2025-08-26 18:16:14,874 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-08-26 18:17:13,852 - INFO - Processing symptom: 'I was crouched and when I rised up, I got dizzy and my vision went black for about 15 seconds.'


1. On when does it last? (Dull or sharp/knife-like, achy or pressure, tingling, etc.)
2. What triggers it? (Drugs or alcohol, stress, exercise, illness, surgery, etc.)
3. How often does it occur? (Several times a day, once a week, once a month, etc.)
4. How severe is it? (Mild to moderate, severe, life-threatening)
5. What causes it? (Symptoms of dehydration, allergies, infections, blood vessel damage, medications, nutrient deficiencies, etc.)
6. What alleviates or worsens it? (Foods to avoid, rest, physical activity, medications, supplements, etc.)


PATIENT'S SYMPTOM: I was crouched and when I rised up, I got dizzy and my vision went black for about 15 seconds.
--------------------------------------------------------------------------------

🤖 Generating questions WITHOUT RAG (from model's general knowledge)...


2025-08-26 18:32:53,042 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Could not parse questions. Raw response:
 QUESTION 1: What symptoms did you experience during the episode? Answer:
- Croqueting disorder, and dizziness, blackouts for about 15 seconds.

QUESTION 2: Did you have any recent medical history or doctor's visits? Answer:
No, but I recently got a prescription for Xanax to help with my anxiety.

QUESTION 3: What do you remember about the episode that led up to your symptoms? Answer:
I was trying to read during my lunch break at work and I started feeling dizzy when I looked down.

QUESTION 4: How has your diet been since the episode? Answer:
I've been eating a lot more vegetables, but not sure if that's because of the episode or just to try new things.

QUESTION 5: Have you been taking any medication for your symptoms? Answer:
Yes, I took Xanax a few weeks ago when my anxiety was getting really bad.

By asking these questions, the medical assistant should be able to gather information about the patient's medical history and doctor's visits tha